In [1]:
import os

In [2]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main\\research'

In [3]:
from pathlib import Path
import os

_cwd = Path.cwd().resolve()
if _cwd.name == "research":
    os.chdir(_cwd.parent)

In [4]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareCallbacksConfig:
    root_dir: Path
    tensorboard_root_log_dir: Path
    checkpoint_model_filepath: Path

In [6]:
import sys
import os
import unittest

if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

sys.path.insert(0, os.path.abspath("src"))

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self, 
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])


    
    def get_prepare_callback_config(self) -> PrepareCallbacksConfig:
        config = self.config.prepare_callbacks
        model_ckpt_dir = os.path.dirname(config.checkpoint_model_filepath)
        create_directories([
            Path(model_ckpt_dir),
            Path(config.tensorboard_root_log_dir)
        ])

        prepare_callback_config = PrepareCallbacksConfig(
            root_dir=Path(config.root_dir),
            tensorboard_root_log_dir=Path(config.tensorboard_root_log_dir),
            checkpoint_model_filepath=Path(config.checkpoint_model_filepath)
        )

        return prepare_callback_config




In [8]:
import os
import urllib.request as request
from zipfile import ZipFile
import time

os.environ.setdefault("KERAS_BACKEND", "torch")
try:
    import tensorflow as tf
except ModuleNotFoundError:
    import keras as _keras

    class _TF:
        keras = _keras

    tf = _TF()

In [9]:
class PrepareCallback:
    def __init__(self, config: PrepareCallbacksConfig):
        self.config = config


    
    @property
    def _create_tb_callbacks(self):
        timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
        tb_running_log_dir = os.path.join(
            self.config.tensorboard_root_log_dir,
            f"tb_logs_at_{timestamp}",
        )
        return tf.keras.callbacks.TensorBoard(log_dir=tb_running_log_dir)
    

    @property
    def _create_ckpt_callbacks(self):
        return tf.keras.callbacks.ModelCheckpoint(
            filepath=str(self.config.checkpoint_model_filepath),
            save_best_only=True,
            monitor="val_loss",
            mode="min",
        )


    def get_tb_ckpt_callbacks(self):
        import importlib.util

        callbacks = []
        if importlib.util.find_spec("tensorflow") is not None:
            callbacks.append(self._create_tb_callbacks)
        callbacks.append(self._create_ckpt_callbacks)
        return callbacks


In [10]:
try:
    config = ConfigurationManager()
    prepare_callbacks_config = config.get_prepare_callback_config()
    prepare_callbacks = PrepareCallback(config=prepare_callbacks_config)
    callback_list = prepare_callbacks.get_tb_ckpt_callbacks()
    
except Exception as e:
    raise e

[2026-03-28 23:11:35,023: INFO: common: yaml file: config\config.yaml loaded successfully]


[2026-03-28 23:11:35,025: INFO: common: yaml file: params.yaml loaded successfully]


[2026-03-28 23:11:35,025: INFO: common: created directory at: artifacts]


[2026-03-28 23:11:35,026: INFO: common: created directory at: artifacts\prepare_callbacks\checkpoint_dir]


[2026-03-28 23:11:35,027: INFO: common: created directory at: artifacts\prepare_callbacks\tensorboard_log_dir]


In [11]:
import time
timestamp = time.strftime("%Y-%m-%d-%H-%M-%S")
f"tb_logs_at_{timestamp}"


'tb_logs_at_2026-03-28-23-11-35'